In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install python-igraph leidenalg pandas pyarrow -q
# import citation_clustering

In [ ]:
#!/usr/bin/env python3
"""
citation_clustering.py  —  Google Colab
=======================================
Build the REPRODUCIBLE citation-based thematic backbone (the CWTS/Leiden
standard) from oa_referenced_works, using BIBLIOGRAPHIC COUPLING: two papers are
linked if they share references. Coupling works entirely within your corpus (no
need for cited papers to be in the set) and handles recent/uncited papers — the
relatedness-comparison literature (Waltman et al. 2020; Ahlgren et al. 2020)
finds it among the most accurate clustering signals for biomedical sets.

Then partition with the Leiden algorithm into clusters = your thematic groups.
This is the deterministic structural layer; BERTopic (separate script) adds
interpretable labels on top. Report BOTH; disclose they capture different
structure (Xie & Waltman 2025).

SCALE: naive all-pairs coupling is O(n^2) (billions of comparisons at 100k+).
We invert it — group papers by shared reference (an inverted index) — which is
fast and memory-light. A min-shared-references threshold keeps the graph
tractable and the edges meaningful.

OUTPUT
  1. citation_clusters.parquet   oa_id -> cluster_id, cluster_size
  2. cluster_top_refs.parquet    per cluster: most-shared references (its "canon")
  3. merged onto corpus: cluster_id column for tensor/equity/trend axes

SETUP
  !pip install python-igraph leidenalg pandas pyarrow -q
"""

import sys
from collections import defaultdict
from itertools import combinations
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/screened_llm_gold.parquet"  # any parquet with oa_id + oa_referenced_works
OUT_CLUSTERS = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/citation_clusters_gold.parquet"
OUT_TOPREFS  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/cluster_top_refs_gold.parquet"
OUT_MERGED   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_clusters_gold.parquet"

SEP = " | "                    # the separator used when harvesting list columns
MIN_SHARED_REFS = 2            # edge only if two papers share >= this many refs
MIN_REFS_PER_PAPER = 3         # ignore papers with too few refs to couple on
RESOLUTION = 1.0               # Leiden resolution: higher -> more, smaller clusters
MIN_CLUSTER_SIZE = 20          # relabel tiny clusters as -1 (noise)
RANDOM_SEED = 42               # Leiden is deterministic given a seed


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def parse_refs(cell):
    return [r.strip() for r in S(cell).split("|") if r.strip()]


# =========================================================================== #
# 1. inverted index: reference -> list of papers that cite it                 #
# =========================================================================== #
def build_edges(df):
    ref_col = "oa_referenced_works"
    papers = {}                                    # oa_id -> set(refs)
    for _, r in df.iterrows():
        refs = set(parse_refs(r.get(ref_col)))
        if len(refs) >= MIN_REFS_PER_PAPER:
            papers[S(r["oa_id"])] = refs
    print(f"[couple] {len(papers):,} papers with >= {MIN_REFS_PER_PAPER} refs",
          file=sys.stderr)

    # inverted index: which papers share each reference
    ref_to_papers = defaultdict(list)
    for pid, refs in papers.items():
        for ref in refs:
            ref_to_papers[ref].append(pid)

    # accumulate shared-reference counts only among papers that co-cite something
    pair_shared = defaultdict(int)
    for ref, plist in ref_to_papers.items():
        if len(plist) < 2:
            continue
        # a hugely-cited reference (e.g. a methods paper) links everything -> skip
        # references shared by >5% of the corpus add noise, not signal
        if len(plist) > 0.05 * len(papers):
            continue
        for a, b in combinations(sorted(plist), 2):
            pair_shared[(a, b)] += 1

    edges = [(a, b, w) for (a, b), w in pair_shared.items() if w >= MIN_SHARED_REFS]
    print(f"[couple] {len(edges):,} coupling edges (>= {MIN_SHARED_REFS} shared refs)",
          file=sys.stderr)
    return papers, edges


# =========================================================================== #
# 2. Leiden partition of the coupling graph                                   #
# =========================================================================== #
def cluster(papers, edges):
    import igraph as ig
    import leidenalg as la

    nodes = sorted(papers.keys())
    idx = {n: i for i, n in enumerate(nodes)}
    g = ig.Graph()
    g.add_vertices(len(nodes))
    g.add_edges([(idx[a], idx[b]) for a, b, _ in edges])
    g.es["weight"] = [w for _, _, w in edges]
    print(f"[leiden] graph: {g.vcount():,} nodes, {g.ecount():,} edges", file=sys.stderr)

    part = la.find_partition(g, la.RBConfigurationVertexPartition,
                             weights="weight", resolution_parameter=RESOLUTION,
                             seed=RANDOM_SEED)
    membership = part.membership
    sizes = pd.Series(membership).value_counts()
    # relabel tiny clusters -> -1
    small = set(sizes[sizes < MIN_CLUSTER_SIZE].index)
    out = pd.DataFrame({"oa_id": nodes,
                        "cluster_id": [(-1 if m in small else m) for m in membership]})
    csz = out["cluster_id"].value_counts().to_dict()
    out["cluster_size"] = out["cluster_id"].map(csz)
    n_real = out.loc[out["cluster_id"] != -1, "cluster_id"].nunique()
    print(f"[leiden] {n_real} clusters (>= {MIN_CLUSTER_SIZE} papers); "
          f"{(out['cluster_id']==-1).sum():,} nodes in noise", file=sys.stderr)
    return out


# =========================================================================== #
# 3. per-cluster canonical references (what defines each cluster)             #
# =========================================================================== #
def cluster_top_refs(papers, clusters, top=15):
    cl = clusters.set_index("oa_id")["cluster_id"].to_dict()
    ref_by_cluster = defaultdict(lambda: defaultdict(int))
    for pid, refs in papers.items():
        c = cl.get(pid, -1)
        if c == -1:
            continue
        for ref in refs:
            ref_by_cluster[c][ref] += 1
    rows = []
    for c, refs in ref_by_cluster.items():
        for ref, cnt in sorted(refs.items(), key=lambda kv: -kv[1])[:top]:
            rows.append({"cluster_id": c, "ref_oa_id": ref, "n_citing": cnt})
    t = pd.DataFrame(rows)
    t.to_parquet(OUT_TOPREFS, index=False)
    print(f"[refs] top references per cluster -> {OUT_TOPREFS}", file=sys.stderr)
    return t


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    papers, edges = build_edges(df)
    clusters = cluster(papers, edges)
    clusters.to_parquet(OUT_CLUSTERS, index=False)
    cluster_top_refs(papers, clusters)

    out = df.merge(clusters, on="oa_id", how="left")
    out["cluster_id"] = out["cluster_id"].fillna(-1).astype(int)
    out.to_parquet(OUT_MERGED, index=False)
    cov = (out["cluster_id"] != -1).mean()
    print(f"[write] {OUT_MERGED}  ({cov:.1%} of papers in a real cluster)", file=sys.stderr)

In [ ]:
out

In [ ]:
cov

In [ ]:


[in] 103,737 rows
[couple] 75,678 papers with >= 3 refs
[couple] 11,740,429 coupling edges (>= 2 shared refs)
[leiden] graph: 75,678 nodes, 11,740,429 edges
[leiden] 16 clusters (>= 20 papers); 3,527 nodes in noise
[refs] top references per cluster -> /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/cluster_top_refs.parquet
[write] /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_clusters.parquet  (69.6% of papers in a real cluster)





[in] 103,737 rows
[couple] 75,678 papers with >= 3 refs
[couple] 11,740,429 coupling edges (>= 2 shared refs)
[leiden] graph: 75,678 nodes, 11,740,429 edges
[leiden] 16 clusters (>= 20 papers); 3,527 nodes in noise
[refs] top references per cluster -> /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/cluster_top_refs.parquet
[write] /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_clusters.parquet  (69.6% of papers in a real cluster)

